# 02 — The seed-noise floor, and the pre-registered gate

**Runtime → GPU → L4** (an A100 is roughly 2× faster and several times the compute units;
either is fine, but **all 11 runs must be on the same card**, and this notebook enforces that).

This is the day that decides whether the project has a paper in it.

**The rule, fixed in `PREREGISTRATION.md` §5 before any model existed:**

```
Delta_model = mean over eyes [ vCDR(model trained on user1) - vCDR(model trained on user7) ]
Delta_ann   = 0.269                      (the annotation-level gap, already measured)
SD_seed     = SD over 5 same-label runs of the per-model mean predicted vCDR

PASS  if  Delta_model > 0.135  (at least half the annotation gap survives training)
     AND  Delta_model > 3 x SD_seed      (it clears the noise floor)
```

**The noise floor runs first, on purpose.** Without it, "models inherit conventions" cannot be
falsified, and 0.135 on its own is an assertion rather than a test.

**If it fails, nothing is re-tuned.** The paper becomes the negative result, reported with the
same prominence. Read that again before you run the cells.

11 runs: roughly 2.5–3.5 h on an L4, 1.5–2 h on an A100. Every run is checkpointed, so a
disconnect costs one run at most.


In [ ]:
# --- environment -----------------------------------------------------------
# WHERE THINGS LIVE, and why:
#   PROJECT  this project's code (src/, analysis/)   -> Drive, or a GitHub clone
#   RAW      the 1.5 GB dataset                      -> LOCAL disk: Drive I/O is slow,
#                                                       and re-fetching costs ~3 minutes
#   WORK     crops, weights, results                 -> Drive, so a disconnect costs nothing
REPO_URL = ""                                    # optional: "https://github.com/<you>/<repo>.git"
DRIVE_PROJECT = "/content/drive/MyDrive/whose-eyes"
WORK = "/content/drive/MyDrive/whose-eyes-work"
RAW = "/content/raw"

import json, os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

if REPO_URL:
    if not os.path.isdir("/content/project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    PROJECT = "/content/project/whose-eyes"
else:
    PROJECT = DRIVE_PROJECT
assert os.path.isdir(os.path.join(PROJECT, "src")), (
    f"No src/ under {PROJECT}. Copy the `whose-eyes` folder into your Drive at "
    f"{DRIVE_PROJECT}, or set REPO_URL above.")
sys.path.insert(0, PROJECT)
for sub in ("crops", "runs", "results"):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
os.makedirs(RAW, exist_ok=True)

# --- GPU ---------------------------------------------------------------------
# Batch size is set from the card, not guessed: U-Net + EfficientNet-B4 at 512 px with AMP
# needs roughly 1.5 GB per sample. Everything else in the schedule stays fixed.
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 24 if VRAM > 60 else 16 if VRAM > 35 else 8 if VRAM > 20 else 4
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    GPU, VRAM, BATCH = "cpu", 0.0, 2

print(f"project : {PROJECT}")
print(f"work    : {WORK}   (Drive - survives a disconnect)")
print(f"raw     : {RAW}   (local - re-fetched each session)")
print(f"gpu     : {GPU}  {VRAM:.0f} GB  ->  batch size {BATCH}")


In [ ]:
!pip -q install "segmentation-models-pytorch==0.4.0" "timm>=1.0.9" || pip -q install segmentation-models-pytorch

import time
import numpy as np, pandas as pd
from dataclasses import replace
from src import train as train_mod, vcdr, labels as labels_mod

CFG = replace(train_mod.CONFIG, batch_size=BATCH)

manifest = pd.read_csv(os.path.join(WORK, "crops", "manifest.csv"), dtype={"image_id": str})
train_rows = manifest[manifest.part == "train"]
val_rows = manifest[manifest.part == "val"]
eval_rows = manifest[manifest.part == "eval"]
print(f"train {len(train_rows)} (Zeiss) | val {len(val_rows)} (Zeiss) | "
      f"eval {len(eval_rows)} (Canon)")
print(f"gpu {GPU} | batch {CFG.batch_size} | epochs {CFG.epochs} | patience {CFG.patience}")
print(f"\nTHE GATE:  Delta_model > 0.135  AND  Delta_model > 3 x SD_seed"
      f"   (Delta_ann = {vcdr.DELTA_ANN})")


In [ ]:
# --- every compared run must sit on the SAME card ---------------------------
# TF32 and cuDNN kernels differ between L4 and A100, so a noise floor measured on one card
# and extremes measured on the other would put a hardware difference inside Delta_model.
# The gate refuses to report across mixed hardware.
def assert_same_gpu(frame, column="gpu"):
    cards = sorted(set(frame[column].dropna()))
    if len(cards) > 1:
        raise RuntimeError(
            "Runs span more than one GPU: " + ", ".join(cards) + ".\n"
            "Delete whose-eyes-work/runs and whose-eyes-work/results/gate_runs.csv and redo "
            "all 11 runs on one card, or the comparison is contaminated.")
    return cards[0] if cards else "unknown"


In [ ]:
# --- one reusable run ---------------------------------------------------------
RESULTS = os.path.join(WORK, "results", "gate_runs.csv")

def load_done():
    return pd.read_csv(RESULTS) if os.path.exists(RESULTS) else pd.DataFrame(
        columns=["label_source", "seed", "gpu", "mean_vcdr"])

def run(label_source: str, seed: int):
    done = load_done()
    tag = f"{label_source}_seed{seed}"
    if len(done) and ((done.label_source == label_source) & (done.seed == seed)).any():
        print(f"{tag}: already done, skipping")
        return
    print(f"\n=== {tag} ===")
    t0 = time.time()
    model, info = train_mod.train_one(
        train_mod.CropDataset(train_rows, label_source, WORK, augment=True),
        train_mod.CropDataset(val_rows, label_source, WORK),
        seed=seed, out_dir=os.path.join(WORK, "runs", tag), config=CFG, log_every=5)
    predicted = train_mod.predict_vcdr(
        model, train_mod.CropDataset(eval_rows, label_source, WORK),
        batch_size=CFG.batch_size)
    np.save(os.path.join(WORK, "runs", tag, "eval_vcdr.npy"), predicted)
    row = {"label_source": label_source, "seed": seed, "gpu": GPU,
           "mean_vcdr": float(np.nanmean(predicted)),
           "refer_at_0.6": vcdr.referral_rate(predicted, 0.6),
           "refer_at_0.7": vcdr.referral_rate(predicted, 0.7),
           "best_val_loss": info["best_val_loss"], "minutes": (time.time() - t0) / 60}
    pd.concat([done, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS, index=False)
    print(f"  mean vCDR {row['mean_vcdr']:.4f} | refer@0.6 {100 * row['refer_at_0.6']:.1f}%"
          f" | {row['minutes']:.1f} min")


In [ ]:
# --- STEP 1: the noise floor. 5 seeds, ONE label. Before anything else. ------
for seed in (101, 102, 103, 104, 105):
    run(labels_mod.NOISE_FLOOR_LABEL, seed)

floor = load_done()
floor = floor[floor.label_source == labels_mod.NOISE_FLOOR_LABEL]
assert_same_gpu(floor)
SD_SEED = float(floor.mean_vcdr.std(ddof=1))
print(f"\nSEED-NOISE FLOOR on '{labels_mod.NOISE_FLOOR_LABEL}' ({len(floor)} seeds, {GPU})")
print(floor[["seed", "mean_vcdr", "refer_at_0.6", "minutes"]].to_string(index=False))
print(f"\n  SD_seed      = {SD_SEED:.4f}")
print(f"  3 x SD_seed  = {3 * SD_SEED:.4f}   <- any effect smaller than this is noise")


In [ ]:
# --- STEP 2: the two extremes, 3 seeds each ----------------------------------
for label_source in labels_mod.EXTREMES:
    for seed in (42, 43, 44):
        run(label_source, seed)

runs = load_done()
card = assert_same_gpu(runs)
summary = runs.groupby("label_source").agg(
    mean_vcdr=("mean_vcdr", "mean"), sd=("mean_vcdr", "std"),
    refer_06=("refer_at_0.6", "mean"), n=("seed", "count"))
print(f"all {len(runs)} runs on {card}\n")
print(summary.round(4).to_string())


In [ ]:
# --- STEP 3: the gate ---------------------------------------------------------
# Delta_model is computed PER EYE and per seed pair, so it is a like-for-like contrast
# rather than the difference of two separately averaged numbers.
def load(label_source, seed):
    return np.load(os.path.join(WORK, "runs", f"{label_source}_seed{seed}", "eval_vcdr.npy"))

per_seed = [float(np.nanmean(load("user1", s) - load("user7", s))) for s in (42, 43, 44)]
DELTA_MODEL = float(np.mean(per_seed))

verdict = vcdr.gate(DELTA_MODEL, SD_SEED)
verdict["delta_per_seed"] = per_seed
verdict["gpu"] = card
verdict["refer_at_0.6"] = {s: float(summary.loc[s, "refer_06"]) for s in summary.index}
json.dump(verdict, open(os.path.join(WORK, "results", "gate.json"), "w"), indent=2)

print("=" * 74)
print(f"  Delta_ann   (annotations)  {verdict['delta_ann']:.4f}")
print(f"  Delta_model (trained)      {verdict['delta_model']:.4f}    per seed: "
      + ", ".join(f"{d:.4f}" for d in per_seed))
print(f"  retention                  {100 * verdict['retention']:.1f}%     (must be >= 50%)")
print(f"  3 x SD_seed                {verdict['noise_floor_3sd']:.4f}"
      f"    (Delta_model must exceed it)")
print(f"  hardware                   {card}")
print("=" * 74)
print(f"  {verdict['verdict']}")
print("=" * 74)


## Whichever way it went

**PASS** — go on to the full grid: the other seven label sources, 3 seeds each (27 more U-Net
runs), then RETFound and DINOv3 on an **A100**. The referral-rate table is the paper's headline.

**FAIL** — stop and write the negative result. Do **not** raise the margin, change the
architecture, swap the evaluation set, or lower 0.135. Each of those is why published findings
fail to replicate, and the pre-registration exists to put them out of reach. A clean negative
is still a contribution: the domain-generalisation literature assumes conventions propagate
through training and has never measured it.

Either way, append the outcome and today's date to `PREREGISTRATION.md` §10.
